<div align="center">

# Predicción de tasas de mortalidad y natalidad a partir de indicadores socioeconómicos

**Proyecto Integrador**

Universidad Internacional del Ecuador — UIDE

---

**Notebook 04 — TASK-10: Optimización de hiperparámetros y evaluación avanzada de métricas**

---

Equipo: Guillermo Paredes · Donato Oña · Mateo Villacreses

Junio 2026

</div>

## Introducción y Objetivos de la TASK-10

En el notebook anterior (*Notebook 03 — Modelado y Evaluación*), entrenamos y evaluamos cuatro modelos base (Ridge, Random Forest, XGBoost y LightGBM) utilizando un esquema de validación cruzada grupal (`GroupKFold`) para predecir la tasa de mortalidad (`death_rate`) y la tasa de natalidad (`birth_rate`). Los resultados demostraron que los modelos basados en árboles superan significativamente al modelo lineal regularizado (Ridge) en la predicción de la mortalidad, confirmando la existencia de la relación no lineal (forma de U invertida) entre el desarrollo socioeconómico y la mortalidad.

Sin embargo, los modelos base se evaluaron utilizando hiperparámetros predeterminados o semi-fijos. El objetivo de este notebook es abordar las dos oportunidades de mejora críticas identificadas en el cierre de la fase de modelado:

1. **Optimización Rigurosa de Hiperparámetros**: Realizar una búsqueda en cuadrícula (`GridSearchCV`) sobre el modelo ganador para la tasa de mortalidad (`death_rate`), respetando estrictamente el esquema de validación cruzada grupal (`GroupKFold` agrupado por país) para evitar filtración de información y asegurar generalización ante países no observados.
2. **Evaluación de Métricas y Diagnóstico Fino**: Generar una comparación de métricas antes/después de la optimización y visualizar la morfología de la mortalidad a través de gráficos de dispersión de alta calidad de valores reales contra predichos.
3. **Análisis de Interpretabilidad Global y Local con SHAP**: Computar valores SHAP (Shapley Additive exPlanations) basados en la teoría de juegos cooperativos para explicar el peso demográfico y socioeconómico de cada variable predictora.
4. **Diagnóstico Estadístico de Residuales**: Analizar la distribución de los errores del modelo optimizado para detectar sesgos sistemáticos (heterocedasticidad o asimetría).

## 1. Optimización del Modelo Ganador mediante GridSearchCV con Validación Grupal

### Justificación Teórica
Para ajustar de forma fina el modelo con mejor desempeño (en este caso, **XGBoost** para `death_rate`), recurrimos a **GridSearchCV** (Búsqueda por Grilla con Validación Cruzada). Los hiperparámetros de un modelo de *Gradient Boosting* controlan la complejidad de los árboles y el ritmo de aprendizaje para evitar el sobreajuste (overfitting):

- `learning_rate` (eta): Escala la contribución de cada nuevo árbol. Un valor más bajo requiere más árboles pero estabiliza el aprendizaje.
- `max_depth`: Limita la profundidad de los árboles individuales. Profundidades altas permiten capturar interacciones complejas pero aumentan el riesgo de memorizar el ruido.
- `subsample` y `colsample_bytree`: Controlan la fracción de observaciones y variables muestreadas al azar para construir cada árbol, induciendo regularización por bagging.

**Prevención del Data Leakage (Fuga de Datos):**
Dado que nuestro dataset es de tipo panel (países a lo largo del tiempo), las observaciones de un mismo país están fuertemente correlacionadas. Si utilizáramos una validación cruzada tradicional, datos del país *A* en el año 2005 estarían en el entrenamiento y datos del mismo país *A* en 2006 estarían en validación, lo que inflaría artificialmente el desempeño. Por ello, pasamos estrictamente el objeto `cv` (que implementa `GroupKFold`) y el argumento `groups=groups` (códigos de países) al método `.fit()`. Esto garantiza que los pliegues de validación se compongan en su totalidad por países completos que el modelo jamás observó durante el entrenamiento.

In [ ]:
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import numpy as np
import pandas as pd

print("=== INICIANDO OPTIMIZACIÓN DE HIPERPARÁMETROS ===")

# 1. Extraer el modelo base XGBoost desde la infraestructura del proyecto
base_xgb = construir_modelos()['XGBoost']

# 2. Definir la grilla de búsqueda de hiperparámetros
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [4, 6, 8],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

print(f"Configuración de la Grilla: {len(param_grid['learning_rate']) * len(param_grid['max_depth']) * len(param_grid['subsample']) * len(param_grid['colsample_bytree'])} combinaciones.")
print(f"Validación cruzada grupal activa: {cv.n_splits} pliegues.")

# 3. Configurar el GridSearchCV
grid_search = GridSearchCV(
    estimator=base_xgb,
    param_grid=param_grid,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1,
    refit=True
)

# 4. Ajustar la grilla pasando los grupos de países para validar de forma estricta
grid_search.fit(X, y_death, groups=groups)

# 5. Reportar resultados
print("\n✅ Búsqueda por grilla completada.")
print("Mejores Hiperparámetros encontrados:")
for param, val in grid_search.best_params_.items():
    print(f"  - {param}: {val}")

print(f"Mejor puntuación de validación (RMSE): {-grid_search.best_score_:.4f}")

## 2. Evaluación del Desempeño y Visualización de la Morfología de la Mortalidad

### Justificación Teórica
Para realizar un diagnóstico riguroso del modelo optimizado en comparación con su línea base, calculamos tres métricas clave utilizando predicciones fuera de pliegue (*Out-Of-Fold* or *OOF*) obtenidas mediante `cross_val_predict`. 

- **RMSE (Error Cuadrático Medio de la Raíz)**: Penaliza de forma más severa los errores grandes. Expresa la desviación típica de los residuales en la misma unidad que el target (defunciones por cada 1,000 habitantes).
- **MAE (Error Absoluto Medio)**: Mide la magnitud promedio de los errores sin importar su dirección. Al no elevar los errores al cuadrado, es más robusto a observaciones atípicas (outliers).
- **R² (Coeficiente de Determinación)**: Representa la proporción de la varianza del target explicada por los predictores socioeconómicos. Es una métrica adimensional útil para la comparación directa.

**Morfología de la Mortalidad (Real vs. Predicho):**
Graficar los valores reales contra las predicciones OOF nos permite observar la estructura de los errores a lo largo del espectro demográfico. Si los puntos se agrupan estrechamente en torno a la diagonal $y = \hat{y}$, el modelo exhibe alta precisión. Colorizar los puntos por región continental permite identificar si existen sesgos geográficos sistemáticos (por ejemplo, si el modelo sistemáticamente subestima la mortalidad en regiones de bajos ingresos debido a dinámicas locales no capturadas).

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Obtener predicciones OOF de la línea base para comparar
y_pred_base = cross_val_predict(base_xgb, X, y_death, groups=groups, cv=cv, n_jobs=-1)
rmse_base = np.sqrt(mean_squared_error(y_death, y_pred_base))
mae_base = mean_absolute_error(y_death, y_pred_base)
r2_base = r2_score(y_death, y_pred_base)

# 2. Obtener predicciones OOF del modelo optimizado
best_xgb = grid_search.best_estimator_
y_pred_opt = cross_val_predict(best_xgb, X, y_death, groups=groups, cv=cv, n_jobs=-1)
rmse_opt = np.sqrt(mean_squared_error(y_death, y_pred_opt))
mae_opt = mean_absolute_error(y_death, y_pred_opt)
r2_opt = r2_score(y_death, y_pred_opt)

print("=== COMPARACIÓN DE DESEMPEÑO (TASAS DE MORTALIDAD) ===")
print(f"Línea Base (Default)  -> RMSE: {rmse_base:.4f} | MAE: {mae_base:.4f} | R²: {r2_base:.4f}")
print(f"Modelo Optimizado     -> RMSE: {rmse_opt:.4f} | MAE: {mae_opt:.4f} | R²: {r2_opt:.4f}")
print(f"Mejora Relativa R²    -> {((r2_opt - r2_base)/r2_base)*100:+.2f}%")

# 3. Configuración estética para visualización de alta calidad
sns.set_theme(style='whitegrid', context='talk')
plt.figure(figsize=(11, 8.5))

# Verificar si el DataFrame original está disponible para la segmentación geográfica
if 'df' in globals() and 'region' in df.columns:
    region_codes = df['region'].values
    unique_regions = np.unique(region_codes)
    colors_palette = sns.color_palette("Set2", len(unique_regions))
    region_colors = dict(zip(unique_regions, colors_palette))
    
    for reg in unique_regions:
        idx = (region_codes == reg)
        plt.scatter(y_death[idx], y_pred_opt[idx], label=reg, alpha=0.6, s=35, edgecolors='none')
    plt.legend(title='Región Geográfica', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=11)
else:
    plt.scatter(y_death, y_pred_opt, color='#2c3e50', alpha=0.5, s=30, edgecolors='none', label='Observaciones')
    plt.legend(loc='lower right')

# Línea de identidad ideal (Real = Predicho)
lims = [min(y_death.min(), y_pred_opt.min()) - 1, max(y_death.max(), y_pred_opt.max()) + 1]
plt.plot(lims, lims, 'r--', alpha=0.8, lw=2, label='Línea de Identidad ($y = \hat{y}$)')

# Detalles del gráfico
plt.title('Morfología de la Mortalidad: Real vs. Predicho (Modelo Optimizado)', fontsize=16, weight='bold', pad=15)
plt.xlabel('Tasa de Mortalidad Real (defunciones por cada 1,000 hab.)', fontsize=13)
plt.ylabel('Tasa de Mortalidad Predicha (defunciones por cada 1,000 hab.)', fontsize=13)

# Cuadro estadístico informativo
stats_text = '\n'.join((
    r'$R^2 = %.4f$' % (r2_opt,),
    r'$RMSE = %.4f$' % (rmse_opt,),
    r'$MAE = %.4f$' % (mae_opt,)
))
props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='lightgray')
plt.gca().text(0.05, 0.95, stats_text, transform=plt.gca().transAxes, fontsize=12,
            verticalalignment='top', bbox=props)

plt.xlim(lims)
plt.ylim(lims)
plt.tight_layout()

# Guardar el gráfico solicitado por el proyecto
plt.savefig('task10_morfologia_mortalidad.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Interpretabilidad Global y Local con SHAP (Shapley Additive exPlanations)

### Justificación Teórica
Los modelos basados en ensambles de árboles como XGBoost son cajas negras de alta precisión pero difícil interpretabilidad. Para abrir esta caja negra de forma matemáticamente consistente, implementamos **SHAP** (Shapley Additive exPlanations). Basado en la teoría de juegos cooperativos, SHAP calcula el aporte marginal de cada variable predictora en la desviación de la predicción respecto a la media global de predicciones.

A diferencia de las importancias tradicionales por permutación o ganancia de impureza (Gini), los valores SHAP satisfacen propiedades teóricas deseables:
- **Aditividad**: Las contribuciones individuales suman la diferencia entre la predicción y el valor base esperado.
- **Consistencia**: Si un modelo se modifica de modo que el aporte de un feature aumenta o se mantiene igual, su valor SHAP no puede disminuir.

**El gráfico de resumen (`shap.summary_plot`):**
Visualiza la distribución del impacto de cada variable. Cada punto en el gráfico representa una observación del dataset:
- La posición en el **eje Y** indica el orden de importancia global de las variables (de mayor a menor).
- La posición en el **eje X** muestra el impacto de SHAP sobre la tasa de mortalidad (valores positivos a la derecha aumentan la mortalidad predicha; valores negativos a la izquierda la disminuyen).
- El **color** representa la magnitud del valor de la variable (rojo indica valor alto; azul indica valor bajo).

Esto nos permite revelar directamente dinámicas de salud pública complejas (por ejemplo, si un menor PIB per cápita [azul] empuja la mortalidad hacia arriba [valores SHAP positivos] de forma asimétrica).

In [ ]:
import shap
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

print("=== COMPUTANDO VALORES SHAP PARA INTERPRETABILIDAD ===")

# 1. Convertir la matriz X a DataFrame para asociar los nombres de variables correctos
if 'FEATURES' in globals():
    X_df = pd.DataFrame(X, columns=FEATURES)
else:
    X_df = pd.DataFrame(X) # Fallback en caso de que FEATURES no esté en el namespace

# 2. Inicializar el TreeExplainer, optimizado para modelos basados en árboles
try:
    # Intentamos instanciar el Explainer general que detecta TreeExplainer automáticamente
    explainer = shap.Explainer(best_xgb, X_df)
    shap_values = explainer(X_df)
except Exception as e:
    print(f"Aviso: Se usará TreeExplainer clásico debido a: {e}")
    explainer = shap.TreeExplainer(best_xgb)
    shap_values = explainer.shap_values(X_df)

# 3. Generar y guardar el summary_plot
plt.figure(figsize=(12, 7.5))
shap.summary_plot(shap_values, X_df, show=False)

plt.title('Interpretabilidad SHAP: Impacto de Indicadores en la Predicción de Mortalidad', fontsize=14, weight='bold', pad=15)
plt.tight_layout()

# Guardar el gráfico de interpretabilidad
plt.savefig('task10_shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Diagnóstico Fino de Sesgos mediante Análisis de Residuales

### Justificación Teórica
El análisis de residuales (definidos como el error de predicción: $\text{Residual} = y - \hat{y}$) es el método más riguroso para auditar la validez estadística de un regresor. Al graficar la distribución de los errores mediante un histograma acoplado a una curva de estimación de densidad por kernel (KDE), podemos verificar los siguientes aspectos:

- **Centralidad y Sesgo (Sesgo Medio $\mu$)**: Una media próxima a cero indica que el modelo no tiene subestimación o sobreestimación sistemática global.
- **Precisión (Desviación Estándar $\sigma$)**: Muestra la dispersión de los errores. Mientras menor sea su amplitud, mayor es la precisión predictiva local del modelo.
- **Asimetría (Skewness)**: Evalúa si los errores se inclinan asimétricamente hacia un lado. Una asimetría positiva larga significa que el modelo subestima gravemente las tasas de mortalidad extremadamente altas (los errores reales son mucho mayores que lo predicho), algo crucial de detectar en escenarios de crisis humanitarias o epidemias.
- **Curtosis**: Muestra la concentración de errores extremos (colas pesadas). Valores de curtosis elevados indican que existen países específicos con características anómalas donde el modelo comete errores masivos.

In [ ]:
from scipy.stats import skew, kurtosis

# 1. Calcular residuales del modelo optimizado (fuera de muestra)
residuales = y_death - y_pred_opt

# 2. Calcular estadísticas de la distribución de errores
mu_error = np.mean(residuales)
std_error = np.std(residuales)
asimetria = skew(residuales)
curtosis_exceso = kurtosis(residuales)

print("=== ANÁLISIS ESTADÍSTICO DE RESIDUALES ===")
print(f"Media del error (Sesgo):          {mu_error:+.4f}")
print(f"Desviación estándar (Precisión):   {std_error:.4f}")
print(f"Coeficiente de Asimetría:          {asimetria:.4f}")
print(f"Curtosis en exceso:                {curtosis_exceso:.4f}")

# 3. Graficar la distribución de los errores con KDE
plt.figure(figsize=(11, 6.5))
sns.histplot(residuales, kde=True, color='#8e44ad', edgecolor='white', bins=50, line_kws={'linewidth': 2.5})

# Línea de referencia en cero (predicción perfecta)
plt.axvline(0, color='#e74c3c', linestyle='--', linewidth=2, label='Ausencia de Sesgo ($e = 0$)')

# Añadir marcadores de la media y la desviación típica
plt.axvline(mu_error, color='#2ecc71', linestyle=':', linewidth=2, label=f'Media: {mu_error:+.3f}')

# Detalles del gráfico
plt.title('Diagnóstico del Modelo: Distribución de Residuales de Mortalidad OOF', fontsize=15, weight='bold', pad=15)
plt.xlabel('Residual (Mortalidad Real - Mortalidad Predicha)', fontsize=12)
plt.ylabel('Frecuencia de Países-Año / Densidad KDE', fontsize=12)

# Cuadro de texto con diagnósticos matemáticos
diag_text = '\n'.join((
    r'$\mu_{error} = %.4f$' % (mu_error,),
    r'$\sigma_{error} = %.4f$' % (std_error,),
    r'$Asimetr\acute{\iota}a = %.4f$' % (asimetria,),
    r'$Curtosis = %.4f$' % (curtosis_exceso,)
))
props_diag = dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='darkviolet')
plt.gca().text(0.05, 0.95, diag_text, transform=plt.gca().transAxes, fontsize=11,
            verticalalignment='top', bbox=props_diag)

plt.legend(loc='upper right')
plt.tight_layout()

# Guardar el gráfico de residuales
plt.savefig('task10_residuales_optimizados.png', dpi=300, bbox_inches='tight')
plt.show()